# Mini reporte y uso de IA
### Masterclass Data Analytics con Python — Academia MC

Con el mismo análisis de ventas podemos preparar un **resumen** y volver a ejecutarlo cuando llegue un archivo nuevo con la misma estructura.

> **Nota:** como en este archivo cada fila representa una venta (un ticket) completo, es correcto llamar "ticket promedio" al monto promedio por registro. Si cada fila representara un producto dentro de un pedido, el cálculo debería adaptarse (agrupando primero por pedido).

In [1]:
import pandas as pd

# Si ventas.csv está en la misma carpeta (corriendo local) lo usamos;
# si no (por ejemplo en Google Colab), lo bajamos directo de GitHub.
import os
URL_CSV = "https://raw.githubusercontent.com/marianacasella92/masterclass-data-analytics/main/ventas.csv"
ventas = pd.read_csv("ventas.csv" if os.path.exists("ventas.csv") else URL_CSV)
ventas['fecha'] = pd.to_datetime(ventas['fecha'], format="%d/%m/%Y")

faltantes = ventas['monto'].isna().sum()
ventas_con_monto = ventas.dropna(subset=['monto']).copy()
ventas_con_monto['mes'] = ventas_con_monto['fecha'].dt.to_period('M')

ventas_por_mes = ventas_con_monto.groupby('mes')['monto'].sum()

## 1. Calcular los tres indicadores

In [2]:
ventas_totales = ventas_con_monto['monto'].sum()
ticket_promedio = ventas_con_monto['monto'].mean()
mes_mayor_venta = ventas_por_mes.idxmax()
monto_mes_mayor_venta = ventas_por_mes.max()

print(f"Ventas totales:         ${ventas_totales:,.0f}")
print(f"Ticket promedio:        ${ticket_promedio:,.0f}")
print(f"Mes de mayor venta:     {mes_mayor_venta}  (${monto_mes_mayor_venta:,.0f})")
print(f"Ventas sin monto:       {faltantes} (excluidas de estos cálculos)")

Ventas totales:         $25,364,600
Ticket promedio:        $14,282
Mes de mayor venta:     2025-12  ($3,636,100)
Ventas sin monto:       74 (excluidas de estos cálculos)


## 2. Pedirle a una IA que redacte una explicación

Ahora podemos pedirle a una IA que nos ayude a redactar una explicación de estos resultados.

Le vamos a dar **solo los valores que calculamos** (el resumen agregado, nunca el archivo completo con datos de clientes) y una instrucción: que **describa** lo que muestran, que **no invente causas**, y que **sugiera qué preguntas** convendría investigar.

> **Para la clase en vivo:** esta celda está preparada como **demo simulada** — no llama a ninguna API en vivo. La salida de abajo es una ejecución previa, guardada como respaldo (más seguro para un directo: sin depender de internet ni de una API externa en el momento). Si en algún momento se arma con una API real (Claude o GPT), la estructura del prompt es la misma; solo cambia cómo se envía.

In [3]:
# --- Prompt que se le enviaría a la IA (esto es lo único que ve el modelo) ---

prompt = f"""
Tengo estos indicadores de un análisis de ventas:

- Ventas totales: ${ventas_totales:,.0f}
- Ticket promedio: ${ticket_promedio:,.0f}
- Mes de mayor venta: {mes_mayor_venta} (${monto_mes_mayor_venta:,.0f})
- Ventas sin monto registrado (excluidas del cálculo): {faltantes}

Redactá una explicación breve de estos resultados para alguien que no es
analista de datos. No inventes causas que los datos no muestran. Al final,
sugerí 2 o 3 preguntas que convendría investigar para entender mejor estos
números.
"""

print(prompt)


Tengo estos indicadores de un análisis de ventas:

- Ventas totales: $25,364,600
- Ticket promedio: $14,282
- Mes de mayor venta: 2025-12 ($3,636,100)
- Ventas sin monto registrado (excluidas del cálculo): 74

Redactá una explicación breve de estos resultados para alguien que no es
analista de datos. No inventes causas que los datos no muestran. Al final,
sugerí 2 o 3 preguntas que convendría investigar para entender mejor estos
números.



In [4]:
# --- Respuesta simulada de la IA (ejecución previa guardada como respaldo) ---
# IMPORTANTE: esta respuesta fue generada una vez y queda fija acá para la demo.
# No se llama a ninguna API en el momento de la clase.

respuesta_ia_simulada = """
En el período analizado, las ventas sumaron $25.364.600, repartidas en
operaciones con un ticket promedio de $14.282 aproximadamente.

El mes con mayor facturación fue diciembre de 2025, con $3.636.100 en ventas,
muy por encima del resto de los meses del período.

Vale aclarar que 74 ventas (4% del total) no tienen un monto registrado y
quedaron fuera de estos cálculos, por lo que el total real podría ser
levemente distinto.

Preguntas para seguir investigando:
1. ¿Qué pasó específicamente en diciembre: hubo una promoción, un cambio de
   precios, o simplemente más operaciones que en otros meses?
2. ¿Por qué faltan montos en esas 74 ventas? ¿Es un problema del sistema de
   carga, o corresponden a un tipo de operación particular (por ejemplo,
   notas de crédito o canjes)?
3. ¿Cómo se compara este período con el mismo período del año anterior, para
   saber si diciembre es un pico estacional habitual o algo puntual de este año?
"""

print(respuesta_ia_simulada)


En el período analizado, las ventas sumaron $25.364.600, repartidas en
operaciones con un ticket promedio de $14.282 aproximadamente.

El mes con mayor facturación fue diciembre de 2025, con $3.636.100 en ventas,
muy por encima del resto de los meses del período.

Vale aclarar que 74 ventas (4% del total) no tienen un monto registrado y
quedaron fuera de estos cálculos, por lo que el total real podría ser
levemente distinto.

Preguntas para seguir investigando:
1. ¿Qué pasó específicamente en diciembre: hubo una promoción, un cambio de
   precios, o simplemente más operaciones que en otros meses?
2. ¿Por qué faltan montos en esas 74 ventas? ¿Es un problema del sistema de
   carga, o corresponden a un tipo de operación particular (por ejemplo,
   notas de crédito o canjes)?
3. ¿Cómo se compara este período con el mismo período del año anterior, para
   saber si diciembre es un pico estacional habitual o algo puntual de este año?



## 3. Verificar la respuesta antes de compartirla

Revisemos: **¿los números coinciden? ¿está afirmando algo que nuestros datos no permiten saber?**

La IA puede ayudarnos a preparar un primer borrador. Como analistas, **tenemos que verificarlo antes de compartirlo**.

In [5]:
# Chequeo rápido: comparar los números que dijo la IA contra los que calculamos nosotros

checks = {
    "Ventas totales ($25.364.600 mencionado)": abs(ventas_totales - 25_364_600) < 1000,
    "Ticket promedio (~$14.282 mencionado)": abs(ticket_promedio - 14_282) < 200,
    "Mes de mayor venta (diciembre 2025 mencionado)": str(mes_mayor_venta) == "2025-12",
    "Monto de diciembre ($3.636.100 mencionado)": abs(monto_mes_mayor_venta - 3_636_100) < 1000,
    "Ventas sin monto (74 mencionado)": faltantes == 74,
}

for afirmacion, coincide in checks.items():
    estado = "✅ coincide" if coincide else "⚠️ REVISAR — no coincide con el dato real"
    print(f"{estado}  —  {afirmacion}")

print()
print("Lo que la IA NO afirma (y está bien que no lo haga): no dice POR QUÉ")
print("subieron las ventas en diciembre. Eso lo señala como pregunta abierta,")
print("no como un hecho — que es justamente lo que le pedimos.")

✅ coincide  —  Ventas totales ($25.364.600 mencionado)
✅ coincide  —  Ticket promedio (~$14.282 mencionado)
✅ coincide  —  Mes de mayor venta (diciembre 2025 mencionado)
✅ coincide  —  Monto de diciembre ($3.636.100 mencionado)
✅ coincide  —  Ventas sin monto (74 mencionado)

Lo que la IA NO afirma (y está bien que no lo haga): no dice POR QUÉ
subieron las ventas en diciembre. Eso lo señala como pregunta abierta,
no como un hecho — que es justamente lo que le pedimos.


---
### Cierre de esta parte

Este mini reporte (los 3 indicadores + la explicación) se puede volver a generar automáticamente
cada vez que llegue un archivo nuevo con la misma estructura — ese es el valor de automatizar
este tipo de análisis con Python.